<a href="https://colab.research.google.com/github/Vishrutha333/BIS_LAB/blob/main/AntColonyOptimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np

points = np.array([
    [50, 50],
    [10, 20],
    [20, 60],
    [30, 40],
    [60, 20],
    [70, 60],
    [80, 30],
    [40, 80],
    [90, 70]
])

demands = [0, 4, 6, 3, 5, 2, 7, 4, 3]

customer_names = {
    0: "Depot",
    1: "Customer 1",
    2: "Customer 2",
    3: "Customer 3",
    4: "Customer 4",
    5: "Customer 5",
    6: "Customer 6",
    7: "Customer 7",
    8: "Customer 8"
}

n_customers = 8
n_vehicles = 2
vehicle_capacity = 20

n_ants = 30
n_iterations = 100

alpha = 1
beta = 2
rho = 0.5
Q = 100

def distance(i, j):
    return np.linalg.norm(points[i] - points[j])

distances = np.zeros((n_customers + 1, n_customers + 1))

for i in range(n_customers + 1):
    for j in range(n_customers + 1):
        distances[i][j] = distance(i, j)

pheromone = np.ones((n_customers + 1, n_customers + 1))

def local_search(route):
    best = route.copy()
    improved = True

    while improved:
        improved = False

        for i in range(1, len(best) - 2):
            for j in range(i + 1, len(best) - 1):

                new_route = best.copy()
                new_route[i:j + 1] = reversed(new_route[i:j + 1])

                old_distance = sum(
                    distances[best[k]][best[k + 1]]
                    for k in range(len(best) - 1)
                )

                new_distance = sum(
                    distances[new_route[k]][new_route[k + 1]]
                    for k in range(len(new_route) - 1)
                )

                if new_distance < old_distance:
                    best = new_route
                    improved = True

    return best

def create_solution():
    unvisited = list(range(1, n_customers + 1))
    routes = []

    for vehicle in range(n_vehicles):

        route = [0]
        current = 0
        load = 0

        while True:

            feasible = [
                c for c in unvisited
                if load + demands[c] <= vehicle_capacity
            ]

            if not feasible:
                break

            probabilities = []

            for customer in feasible:

                tau = pheromone[current][customer] ** alpha

                eta = (1 / (distances[current][customer] + 1e-10)) ** beta

                probabilities.append(tau * eta)

            probabilities = np.array(probabilities)
            probabilities /= np.sum(probabilities)

            next_customer = int(
                np.random.choice(feasible, p=probabilities)
            )

            route.append(next_customer)
            unvisited.remove(next_customer)

            load += demands[next_customer]
            current = next_customer

        route.append(0)

        route = local_search(route)

        routes.append(route)

    if unvisited:
        return None

    return routes

def calculate_total_distance(routes):

    total = 0

    for route in routes:
        for i in range(len(route) - 1):
            total += distances[route[i]][route[i + 1]]

    return total

best_routes = None
best_distance = float('inf')

for iteration in range(n_iterations):

    iteration_best_routes = None
    iteration_best_distance = float('inf')

    for ant in range(n_ants):

        routes = create_solution()

        if routes is None:
            continue

        total_distance = calculate_total_distance(routes)

        if total_distance < iteration_best_distance:

            iteration_best_distance = total_distance
            iteration_best_routes = routes

    if iteration_best_routes is not None:

        if iteration_best_distance < best_distance:

            best_distance = iteration_best_distance
            best_routes = iteration_best_routes

        pheromone *= (1 - rho)

        for route in iteration_best_routes:

            for i in range(len(route) - 1):

                a = route[i]
                b = route[i + 1]

                pheromone[a][b] += Q / iteration_best_distance
                pheromone[b][a] += Q / iteration_best_distance

if best_routes is None:

    print("No feasible solution found.")

else:

    print("BEST ROUTES FOUND")
    print("------------------")

    for i, route in enumerate(best_routes):

        path = " -> ".join(
            customer_names[node] for node in route
        )

        route_distance = sum(
            distances[route[j]][route[j + 1]]
            for j in range(len(route) - 1)
        )

        vehicle_load = sum(
            demands[node] for node in route
        )

        print("\nVehicle", i + 1)
        print("Path:", path)
        print("Load:", vehicle_load, "/", vehicle_capacity)
        print("Distance:", round(route_distance, 2))

    print("\nTotal Distance:", round(best_distance, 2))


BEST ROUTES FOUND
------------------

Vehicle 1
Path: Depot -> Customer 6 -> Customer 4 -> Customer 1 -> Customer 3 -> Depot
Load: 19 / 20
Distance: 159.06

Vehicle 2
Path: Depot -> Customer 2 -> Customer 7 -> Customer 8 -> Customer 5 -> Depot
Load: 15 / 20
Distance: 155.62

Total Distance: 314.68
